# v9: learning something NEW with part of the self switched off
**Run All.** One GPU ("0"). Results: `analysis/v9/newtask/`. Every cell prints as it goes; stop any time, re-running skips what is done.

**Question:** when 10% of each layer's self neurons are switched off, does the model learn a new task worse than when the same number of look-alike task neurons are switched off?

- Start models: the last checkpoint of the distant run (hi→zh→ar) and of the related run (hi→gu→pa), plus a fresh random model as a control.
- New tasks (near → far from translation): English→Indonesian, sentiment (positive/negative), topic (world/sports/business/science), addition ("4 7 + 8 5" → "1 3 2").
- Conditions (held during the whole training): nothing off / 10% self off / 10% look-alike task off / 10% random task off.
- Order: seed 0 of everything first, then seed 1, then seed 2. So if you stop early, every task and condition still has a result.

In [ ]:
import os, sys, glob, json, time, subprocess
os.environ["HF_HUB_DISABLE_XET"] = "1"
GPU = "0"
REPO = "/workspace/shreshth"; os.chdir(REPO); sys.path.insert(0, REPO)
from IPython.display import Image, display
from huggingface_hub import HfApi, hf_hub_download
for attempt in range(5):
    try:
        api = HfApi(); HF_REPO = api.whoami()["name"] + "/cml-lang-self-checkpoints"; FILES = api.list_repo_files(HF_REPO); break
    except Exception as e:
        print("HF retry", attempt, type(e).__name__); time.sleep(10)

def get(run, phases):
    for ph in phases:
        if glob.glob(f"runs/{run}/ckpt/p{ph:03d}_*.pt"):
            continue
        name = [f for f in FILES if f.startswith(f"{run}/ckpt/p{ph:03d}_")]
        if name:
            hf_hub_download(HF_REPO, name[0], local_dir="runs")
        else:
            print("missing (not here, not on HF):", run, ph)

def last_phase(run):
    phs = [int(os.path.basename(f)[1:4]) for f in FILES if f.startswith(f"{run}/ckpt/p")]
    phs += [int(os.path.basename(f)[1:4]) for f in glob.glob(f"runs/{run}/ckpt/p*.pt")]
    return max(phs)

def run(langset, *cmd):
    env = {**os.environ, "CML_LANGSET": langset}
    p = subprocess.Popen([sys.executable, "-u", "newtask_v9.py", "--gpu", GPU, *cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    for line in p.stdout:
        print(line, end="")
    p.wait(); print("exit code", p.returncode)

DIST, REL = "v4b_continual_s0", "v8_indic_continual_s0"
PD, PR = last_phase(DIST), last_phase(REL)
get(DIST, [PD]); get(REL, [PR])
print(f"start models: distant = {DIST} phase {PD} | related = {REL} phase {PR}")

In [ ]:
# check that the files the script needs are on this machine
need = ["tokenizer_v4/spm.model", "tokenizer_v8/spm.model"]
need += [f"data/flores/dev.{l}.jsonl" for l in ["en", "hi", "zh", "ar", "gu", "pa"]]
need += [f"data/en-{l}/val.jsonl" for l in ["hi", "zh", "ar", "gu", "pa"]]
miss = [f for f in need if not os.path.exists(f)]
print("all there" if not miss else f"MISSING: {miss}  -> run your download snippet first, then re-run this cell")

In [ ]:
# new-task data (a few minutes the first time; skips what exists)
subprocess.run([sys.executable, "-u", "prep_newtasks.py"], check=True)

In [ ]:
# seed 0 of everything (prints the time per run and an estimate for what is left)
run("distant", "--start", f"distant={DIST}:{PD}", "--seeds", "0")
run("distant", "--start", "fresh", "--seeds", "0")
run("indic", "--start", f"related={REL}:{PR}", "--seeds", "0")

In [ ]:
# look at seed 0 now (CPU only)
subprocess.run([sys.executable, "newtask_v9.py", "--summarize"])
for f in ["fig1_learning_curves_loss.png", "fig2_learning_curves_score.png", "fig3_near_to_far.png"]:
    display(Image(f"analysis/v9/newtask/{f}"))

In [ ]:
# seeds 1 and 2
for s in ["1", "2"]:
    run("distant", "--start", f"distant={DIST}:{PD}", "--seeds", s)
    run("distant", "--start", "fresh", "--seeds", s)
    run("indic", "--start", f"related={REL}:{PR}", "--seeds", s)

In [ ]:
# final summary
subprocess.run([sys.executable, "newtask_v9.py", "--summarize"])
for f in ["fig1_learning_curves_loss.png", "fig2_learning_curves_score.png", "fig3_near_to_far.png"]:
    display(Image(f"analysis/v9/newtask/{f}"))